# Transformers internal representation

---

## Learning Outcomes

By the end of this session, participants will be able to:
- download a pretrained protein transformer
- predict a masked amino acid
- extract and visualize embeddings
- extract and visualize attention weights

## credits

This notebooks is adapted from [Hari Ramadasan's ELIXIR ML/AI 2026 workshop materials](https://github.com/hramdasan/elixir_summerschool_26/tree/main)



In this notebook, we will download a pretrained protein transformer, predict a masked amino acid, and visualize its embedding and attention weights.

We use [ESM-2](https://huggingface.co/facebook/esm2_t6_8M_UR50D), a model pretrained using masked-residue prediction. The selected version has six transformer layers and approximately eight million parameters. It is small enough to run on a CPU.

## Install and import required packages

Uncomment and run the cell below if needed.

In [ ]:
#!pip install -q torch numpy pandas scikit-learn matplotlib seaborn ipywidgets transformers

In [ ]:
# Special google colab setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Setup complete. Continue with the cells below.")


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Downloading and loading ESM-2

The tokenizer converts amino-acid letters into token IDs. The model converts those tokens into contextual representations and scores for predicting masked residues.

The model we will play with is [facebook/esm2_t6_8M_UR50D](https://huggingface.co/facebook/esm2_t6_8M_UR50D), which is a small (8Million parameters) protein model train 
You can read more in their [accompanying paper](https://www.biorxiv.org/content/10.1101/2022.07.20.500902v2)

We load a fixed model revision so that the weights do not change between runs. `eval()` disables training-time dropout. We also request `attn_implementation="eager"` so the model returns the explicit attention matrices needed for the visualization; some optimized attention implementations do not return these weights.

In [ ]:
%%time
MODEL_ID = "facebook/esm2_t6_8M_UR50D"
REVISION = "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, cache_dir="model_cache")
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID, revision=REVISION, cache_dir="model_cache", attn_implementation="eager"
).to(device).eval()

print("Layers:", model.config.num_hidden_layers)
print("Attention heads per layer:", model.config.num_attention_heads)
print("Features per residue:", model.config.hidden_size)
print("Position representation:", model.config.position_embedding_type)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Basic model usage

Let's first see simple usage pattern: sequence --> Tokenize --> Forward pass

In [ ]:
# uncharacterized protein KAJ6854364.1 from gene M6B38_101410 of Iris pallida
# which has been published after our model
# https://www.ncbi.nlm.nih.gov/datasets/gene/GCA_029216955.1/

sequence = "MSPSLPCTTRPRTSNHQSNRRVRRRGRTTPFLIPPASIQYNNQRVQPRVRPRASVSVYQYQSSSSNPLHQSTQGISCATSGHDSESKKFTQTLTDSDLQSHPPQRRPDDAPEAVVTVPQRPHRRAFPTRPVPTELLPPAVLESYACAYFPGGTRPNLESPHRWDGEPPPVPSPRTTPDPATTPLVLDRRSSVRSVPSSHLSPSCPDLDTERAACLFRQLRTLDFPIPLR"
len( sequence )

In [ ]:
encoded = tokenizer(sequence, return_tensors="pt", return_special_tokens_mask=True).to(device)
encoded

In [ ]:
encoded['input_ids'].shape

In [ ]:
output = model(**encoded)
output

In [ ]:
output['logits'].shape

 * 1 sequence
 * 231 positions (2 special tokens + 24 amino acids)
 * 33 output dimensions = 1 per possible token value

The "logits" correspoond to the score of each token for each position in the given sequence(s).

In [ ]:
encoded['input_ids'].shape

## Predicting a masked amino acid

Pretraining asked the model to predict hidden residues from the surrounding sequence. We can try the same operation by replacing one amino acid with `<mask>`.

The model outputs one score per vocabulary token at each position. Softmax converts the scores at the masked position into a probability distribution. These probabilities describe the model's sequence predictions, not the probability of a biological function or a mutation being safe.

In [ ]:
## created a sequence with a masked armino-acid
masked_position =  50

masked_sequence = sequence[:masked_position] + tokenizer.mask_token + sequence[masked_position + 1:]

print( f"...{masked_sequence[(masked_position-5):(masked_position+15)]}...")
print("masked amino-acid:" , sequence[masked_position])

In [ ]:
inputs = tokenizer(masked_sequence, return_tensors="pt").to(device)
with torch.inference_mode():
        output = model(**inputs)

In [ ]:
## we get the logit of the masked position:
output.logits[0, masked_position+1] # +1 because we added a special token at the beginning of the sequence

In [ ]:
# use softmax to transform these scores into probabilities
probabilities = torch.softmax(output.logits[0, mask_position], dim=-1)
probabilities

In [ ]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
amino_acid_ids = tokenizer.convert_tokens_to_ids(amino_acids)
table = pd.DataFrame({"amino_acid": amino_acids,
                      "probability": probabilities[amino_acid_ids].cpu().numpy()})

print("Original residue:", sequence[masked_position])
table.sort_values("probability", ascending=False).head()

We can pack the whole process in a function:

In [ ]:
def predict_mask(sequence, residue_position, top_k=5):
    # residue_position counts the amino acids from zero, before special tokens are added.
    if not 0 <= residue_position < len(sequence):
        raise ValueError("Choose a residue position within the sequence.")
    masked_sequence = sequence[:residue_position] + tokenizer.mask_token + sequence[residue_position + 1:]
    inputs = tokenizer(masked_sequence, return_tensors="pt").to(device)
    mask_position = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    with torch.inference_mode():
        output = model(**inputs)
    probabilities = torch.softmax(output.logits[0, mask_position], dim=-1)
    # Display the top standard amino acids, retaining their original vocabulary probabilities.
    amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
    amino_acid_ids = tokenizer.convert_tokens_to_ids(amino_acids)
    table = pd.DataFrame({"amino_acid": amino_acids,
                          "probability": probabilities[amino_acid_ids].cpu().numpy()})
    print("Masked input:", masked_sequence)
    print("Original residue:", sequence[residue_position])
    return table.sort_values("probability", ascending=False).head(top_k)

display(predict_mask(sequence, residue_position=10))

**Micro-exercise:** Change the masked position and compare the predictions. Is the original residue always the highest-scoring one? Try a sequence of your own.

In [ ]:
residue_position = 15  # Change this to another valid position.
display(predict_mask(sequence, residue_position))

In [ ]:
# Write your answer here.

## Extracting embeddings

### Residue-level embeddings

The model also returns can also the high-level representations corresponding to the output of some of their hidden layers: the **embeddings**. 

There is one representation for every token. 

> For this we use the encoder inside the masked-language model. It does not need masked-token prediction scores or attention matrices, so those outputs are disabled here to save memory. We keep the transformer frozen.

In [ ]:
# human CDK2 https://www.uniprot.org/uniprotkb/P24941/entry#sequences
sequence="MENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVFEFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYTHEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSFPKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL"

inputs = tokenizer(sequence, padding=True, return_special_tokens_mask=True,
                           return_tensors="pt").to(device)

with torch.inference_mode():
    hidden = model.esm(**inputs, output_attentions=False, output_hidden_states=False).last_hidden_state

In [ ]:
hidden.shape

 * 1 sequence
 * 300 input tokens
 * 320 embedding dimensions

In [ ]:
from sklearn.decomposition import PCA

pca = PCA( )
hidden_pca = pca.fit_transform(  hidden[0].cpu() )
pca.explained_variance_ratio_[:10]

In [ ]:
import seaborn as sns
sns.scatterplot( x = hidden_pca[:,0] , y = hidden_pca[:,1] , hue = list(range(hidden.shape[1])) )

We can try to see if there is a relation to annotated sites in the protein:

In [ ]:
annotation = pd.Series( [""]*len(inputs['input_ids'][0]) )
annotation[0] = "CLS"

annotation[9] = "CDK7 binding"
annotation[88:90] = "CDK7 binding"
annotation[166] = "CDK7 binding"

annotation[10:19] = "ATP binding"
annotation[ [33,86,45] ] = "ATP binding"
annotation[ 81:84 ] = "ATP binding"
annotation[ 129:132 ] = "ATP binding"

annotation[ [127] ] = "active site"
sns.scatterplot( x = hidden_pca[:,0] , y = hidden_pca[:,1] , hue = annotation )

### Sequence-level embeddings

From here, we can **pool the per-residue embedding into a sequence embedding**.

| Strategy | How It Works | Best For | Pros & Cons |
| :--- | :--- | :--- | :--- |
| **Mean Pooling** | Calculates the **average** of all token vectors in the sequence. | General sequence embedding | ➕ Captures the overall context smoothly.<br>➖ Can dilute highly specific, niche signals. |
| **Max Pooling** | Extracts the **maximum value** across each individual dimension out of all tokens. | Niche signal (ie. active site, identifying specific domains. | ➕ Excellent at capturing the strongest, most distinct features.<br>➖ Tends to produce uncalibrated or distorted semantic distances. |
| **First Token (`[CLS]`)** | Uses the vector of the **very first token** (often a special classification token like `[CLS]`) as the summary. | Classification tasks or models explicitly trained on CLS targets. | ➕ Super fast; requires zero mathematical calculation over the sequence.<br>➖ Performs poorly if the model wasn't natively optimized for it. |

> It is also possible to perform a sliding-window pooling to get a "domain-level" embedding.

With ESM2, is common to use either the `[CLS]` token or to do mean pooling



In [ ]:
## using the first token
CLS_embed = hidden[:,0,:]

In [ ]:
## mean pooling

# create a mask for the residue: attention==1 AND not a special token
residue_mask = inputs['attention_mask'] * (1-inputs['special_tokens_mask'])
# expand the mask to the embedding's dimension
residue_mask = residue_mask.unsqueeze(-1).expand(hidden.size()).to(bool)
# compute the average
mean_embed = (hidden*residue_mask).sum( 1 ) / residue_mask.sum(1)

In [ ]:
CLS_pca = pca.transform( CLS_embed )
mean_pca = pca.transform( mean_embed )


sns.scatterplot( x = hidden_pca[:,0] , y = hidden_pca[:,1] , color = "grey" )
plt.scatter( CLS_pca[:,0] , CLS_pca[:,1] , c = 'green' , marker = 'x' , s = 100 , label = 'CLS embed' )
plt.scatter( mean_pca[:,0] , mean_pca[:,1] , c = 'blue' , marker = 'x' , s = 100 , label = 'mean-pooling embed' )
plt.legend()

**Additional considerations:**

 * different pooling strategies can be tested
 * it is typical to focus on the last layer's embedding, but [Kumar & Jha 2025 - preprint](https://arxiv.org/pdf/2512.00376) suggest:
     * earlier layers may hold more information (CLS pooling does work so well with them BTW)
     * pooling embeddings across layer is useful





**Micro-exercise:** Compute the sequence-level embedding for the following sequence using CLS pooling:

In [ ]:
k1 = CLS_embed

In [ ]:
## human insulin
insulin="MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

inputs = tokenizer(insulin, padding=True,
                           return_tensors="pt").to(device)


In [ ]:
## write your solution here


## Classification with embeddings

### loading and embedding the data

We are going to load a CSV file containing 120 reviewed UniProt entries from E. coli K-12: 
 
- 60 are annotated as cytoplasmic and 60 as membrane-associated
- Labels are **0 = cytoplasm, 1 = membrane**.
- Only sequence-derived features are used as model inputs.

The data have predefined training, validation and test partitions. Detected similar sequences were grouped before splitting; each group appears in only one partition. The annotations and selection procedure are described in `data/README.md`.

This is a small teaching dataset. Membrane annotation includes different kinds of membrane association, residual homology may remain, and ESM-2 may have seen these sequences or their relatives during pretraining. The results therefore do not establish generalization to unseen protein families.

In [ ]:
## accounting for the eventual absence of the file (eg, google colab execution)
data_path = Path("../cytoplasm_membrane_data/proteins.csv")
data_path = data_path if data_path.is_file() else None

if data_path is None:
    from urllib.request import urlopen

    # A fixed repository revision keeps the teaching dataset reproducible.
    data_url = "https://raw.githubusercontent.com/hramdasan/elixir_summerschool_26/57c6a252e305256756949986951b66a2cc1f3200/data/proteins.csv"
    
    ## alternative url from my repo - uncomment this when we make the repo public
    #data_url = "https://raw.githubusercontent.com/sib-swiss/bio-llm-training/refs/heads/main/cytoplasm_membrane_data/proteins.csv?token=GHSAT0AAAAAADVEFOLEDHVQJ564YJSPL32I2XJWTXQ"
    data_path = Path("data/proteins.csv")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(data_url, timeout=30) as response:
        csv_bytes = response.read()
    data_path.write_bytes(csv_bytes)
    print("Downloaded proteins.csv into the data folder.")


proteins = pd.read_csv(data_path)
display(proteins[["accession", "location", "length", "split"]].head())
display(pd.crosstab(proteins.split, proteins.location))

In [ ]:
## we compute masks for the different sets
train = proteins.split.eq("train").to_numpy()
val = proteins.split.eq("val").to_numpy()
test = proteins.split.eq("test").to_numpy()
y = proteins.label.to_numpy()

Next we embed the proteins

In [ ]:
def embed_sequences(sequences, batch_size=32):
    vectors = []
    for start in range(0, len(sequences), batch_size):
        inputs = tokenizer(list(sequences[start:start + batch_size]), padding=True,
                           return_tensors="pt").to(device)
        with torch.inference_mode():
            hidden = model.esm(**inputs, output_attentions=False, output_hidden_states=False).last_hidden_state
            
            pooled = hidden[:,0,:] ## CLS pooling
        vectors.append(pooled.cpu().numpy())
    return np.concatenate(vectors).astype(np.float32)



In [ ]:
# Calculate the embeddings now using the downloaded model.
protein_vectors = embed_sequences(proteins.sequence.tolist())
print("Calculated vectors:", protein_vectors.shape)

## amino-acid composition baseline feature

Amino-acid composition gives one fraction for each of the 20 standard residues. It ignores order but may already be useful for membrane association. 


In [ ]:

def amino_acid_composition(sequences):
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    return np.array([[seq.count(aa) / len(seq) for aa in amino_acids] for seq in sequences], dtype=np.float32)

features = {"Amino-acid composition": amino_acid_composition(proteins.sequence),
            "ESM-2 embeddings": protein_vectors}


In [ ]:
features['Amino-acid composition']

## Fitting logistic regression models on both sets of feature

We will compare the amino-acid composition featureset with the 320-dimensional ESM-2 vectors using the same logistic-regression classifier.

Scaling is fitted on the training partition only. The regularization parameter `C` is selected using validation ROC-AUC; smaller C means stronger regularization. We do not use the test data to select it.

**Micro-exercise:**  
 * Add other C values (between $10^{-2}$ and $10^{2}$) and select each representation’s classifier using validation.
 * Which feature set has the better validation score?

In [ ]:
%%time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay


candidate_C = [0.1] ## <-- add more C values
selected = {}
validation_results = []

for name, x in features.items(): # testing the two feature sets
    best_score = -np.inf

    for C in candidate_C: # testing different values for C
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=42))
        
        classifier.fit(x[train], y[train]) # fit with the training set
        
        score = roc_auc_score(y[val], classifier.predict_proba(x[val])[:, 1]) # evaluate with the validation set
        validation_results.append({"representation": name, "C": C, "validation_ROC_AUC": score})
        if score > best_score:
            selected[name] = classifier
            best_score = score
            

In [ ]:
pd.DataFrame(validation_results)

## Evaluating the selected classifier

Each feature set has now been selected without consulting the test data. 

We will now evaluate the final model. The threshold of 0.5 is fixed before evaluation.

There are only 24 test proteins, so one error changes accuracy by about four percentage points. Interpret the confusion matrices alongside the scores.

In [ ]:
test_results = []

name = "ESM-2 embeddings"
x = features["ESM-2 embeddings"]


probability = selected[name].predict_proba(x[test])[:, 1]
predicted = probability >= 0.5
test_results.append({"representation": name,
                     "accuracy": accuracy_score(y[test], predicted),
                     "balanced_accuracy": balanced_accuracy_score(y[test], predicted),
                     "ROC_AUC": roc_auc_score(y[test], probability)})

ConfusionMatrixDisplay.from_predictions(y[test], predicted,
    display_labels=["cytoplasm", "membrane"], cmap="Blues", colorbar=False)
plt.title(name)

display(pd.DataFrame(test_results).set_index("representation").round(3))

**Questions:** 
 * Which parameters were updated in the classification task?
 * What additional work would be needed to fine-tune the transformer?
 * What can this small test set tell us about proteins from another organism?

*Write your answer here.*

## Extracting attention weights

Setting `output_attentions=True` during the forward pass returns the attention weights for every layer and head.

There is one tensor per layer, with shape `(batch, heads, query positions, key positions)`. 
Each row describes how one query position combines information from key/value positions. The model uses bidirectional attention, so a position can read both earlier and later positions.


In [ ]:
## human insulin
insulin="MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

In [ ]:
# Use a short sequence so every token label is readable in the plot.
attention_sequence = insulin
if len(attention_sequence) > 150:
    raise ValueError("Use at most 150 residues for this visualization; shorter sequences give readable plots.")
attention_inputs = tokenizer(attention_sequence, return_tensors="pt").to(device)
attention_tokens = tokenizer.convert_ids_to_tokens(attention_inputs["input_ids"][0].cpu().tolist())

with torch.inference_mode():
    attention_output = model(**attention_inputs, output_attentions=True, output_hidden_states=True)

# Stack layer tensors after removing the one-example batch axis.
attention = torch.stack([layer[0].cpu() for layer in attention_output.attentions])
print("Attention shape (layers, heads, queries, keys):", tuple(attention.shape))
print("Row sums:", attention[0, 0].sum(dim=-1))
assert torch.allclose(attention.sum(dim=-1), torch.ones_like(attention.sum(dim=-1)), atol=1e-5)

### Visualizing one head

In the heatmap, rows are query positions and columns are key positions. Darker squares have larger weights. The red horizontal line marks the selected query; the bar chart shows that same row in more detail.

All tokens, including `<cls>` and `<eos>`, are shown so each row retains its complete probability mass. Token index 0 is `<cls>`; the first residue is at token index 1. The color scale is fixed from 0 to 1 so different heads remain comparable.

A diagonal pattern means positions attend strongly to themselves or their neighbors. Attention to a distant position means the model draws on that position in this layer and head. Attention weights alone do not establish physical contacts, causality or biological importance.

In [ ]:
def plot_attention(layer=0, head=0, query=1):
    weights = attention[layer, head].numpy()
    labels = [f"{i}:{token}" for i, token in enumerate(attention_tokens)]
    n = len(labels)
    fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1]})
    image = ax_map.imshow(weights, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    step = max(1, math.ceil(n / 28))
    ticks = np.arange(0, n, step)
    ax_map.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_map.set_yticks(ticks, [labels[i] for i in ticks], fontsize=8)
    ax_map.axhline(query, color="crimson", linewidth=1)
    ax_map.set(xlabel="Key position (supplies information)", ylabel="Query position (receives information)",
               title=f"Layer {layer + 1}, head {head + 1}")
    fig.colorbar(image, ax=ax_map, fraction=0.046, label="Attention weight")
    ax_bar.bar(np.arange(n), weights[query], color="#277DA1")
    ax_bar.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_bar.set(xlabel="Key position", ylabel="Attention weight", ylim=(0, 1),
               title=f"Query {labels[query]}: one row of the heatmap")
    plt.tight_layout()
    plt.show()

plot_attention(layer=0, head=0, query=1)

### Comparing layers, heads and query positions

The controls below change the view without rerunning the transformer. Layers and heads are displayed starting at 1; the underlying tensors use zero-based indexing. The query menu uses the token indices printed in the table.

If your notebook viewer cannot display widgets, change the arguments to `plot_attention` in the previous cell. The static heatmap and bar chart work without widgets.

In [ ]:
import ipywidgets as widgets

layer_control = widgets.Dropdown(options=[(str(i + 1), i) for i in range(attention.shape[0])],
                                 value=0, description="Layer:")
head_control = widgets.Dropdown(options=[(str(i + 1), i) for i in range(attention.shape[1])],
                                value=0, description="Head:")
query_control = widgets.Dropdown(options=[(f"{i}: {token}", i) for i, token in enumerate(attention_tokens)],
                                 value=1, description="Query:")
attention_view = widgets.interactive_output(plot_attention,
    {"layer": layer_control, "head": head_control, "query": query_control})
display(widgets.HBox([layer_control, head_control, query_control]), attention_view)

**Micro-exercise:** Compare an early layer with a later layer and at least two heads. Find a head that emphasizes nearby positions, special tokens or a broader set of positions. Do all heads show the same pattern?

In [ ]:
plot_attention(layer=0, head=0, query=1)  # Change layer, head or query.

# Annex: attention on human language

https://mbrenndoerfer.com/writing/attention-visualization

In [ ]:
%%time
import torch
from transformers import AutoModel, AutoTokenizer

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

# Example sentence
text = "The cat eats the mouse."
inputs = tokenizer(text, return_tensors="pt")

# Forward pass with attention output
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# Extract attention weights: tuple of (num_layers,) tensors
# Each tensor: (batch=1, num_heads, seq_len, seq_len)
attention_weights = outputs.attentions

# Get tokens for display
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
num_layers = len(attention_weights)
num_heads = attention_weights[0].shape[1]
seq_len = attention_weights[0].shape[2]

In [ ]:
# Extract layer 5, head 0 (zero-indexed)
layer_idx = 5
head_idx = 0

# Shape: (seq_len, seq_len)
attn_matrix = attention_weights[layer_idx][0, head_idx].numpy()

In [ ]:
def plot_attention(layer=0, head=0, query=1):
    weights = attention_weights[layer][0, head].numpy()
    labels = [f"{i}:{token}" for i, token in enumerate(tokens)]
    n = len(labels)
    fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1]})
    image = ax_map.imshow(weights, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    step = max(1, math.ceil(n / 28))
    ticks = np.arange(0, n, step)
    ax_map.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_map.set_yticks(ticks, [labels[i] for i in ticks], fontsize=8)
    ax_map.axhline(query, color="crimson", linewidth=1)
    ax_map.set(xlabel="Key position (supplies information)", ylabel="Query position (receives information)",
               title=f"Layer {layer + 1}, head {head + 1}")
    fig.colorbar(image, ax=ax_map, fraction=0.046, label="Attention weight")
    ax_bar.bar(np.arange(n), weights[query], color="#277DA1")
    ax_bar.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=8)
    ax_bar.set(xlabel="Key position", ylabel="Attention weight", ylim=(0, 1),
               title=f"Query {labels[query]}: one row of the heatmap")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_attention(layer=0, head=5, query=1)